# 2. 数据清洗

本notebook对当当网Python图书原始数据进行清洗处理。

In [71]:
import pandas as pd
import numpy as np
import re
import unicodedata
from pathlib import Path
import os

# =========================
# 1. 读取数据
# =========================
file_path = "../data_raw/python_books_raw.csv"

def read_csv_auto(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"找不到文件：{os.path.abspath(path)}\n请确认脚本运行的目录下是否存在 data_raw 文件夹。")

    encodings = ["utf-8-sig", "utf-8", "gb18030", "gbk"]
    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc, low_memory=False)
        except UnicodeDecodeError:
            continue
        except Exception as e:
            raise Exception(f"读取过程中发生错误: {e}")
            
    raise ValueError("CSV读取失败：所有尝试的编码（UTF-8, GBK等）均无法解析该文件。")

try:
    df = read_csv_auto(file_path).copy()
    print(f"✅ 成功读取文件，共有 {len(df)} 行数据。")
except Exception as e:
    print(f"❌ 读取失败：{e}")
    import sys
    sys.exit()

✅ 成功读取文件，共有 154 行数据。


In [72]:
# =========================
# 2. 列名标准化
# =========================
df.columns = [str(c).strip() for c in df.columns]

rename_map = {}
for col in df.columns:
    c = col.replace("（", "(").replace("）", ")").strip()
    if c in ["原价(元)", "原价"]:
        rename_map[col] = "原价"
    elif c in ["折后价(元)", "折后价"]:
        rename_map[col] = "折后价"
    elif c in ["出版年份", "年份"]:
        rename_map[col] = "出版年份"
    elif c in ["评论数", "评论"]:
        rename_map[col] = "评论数"
    elif c in ["详情链接", "链接", "商品链接"]:
        rename_map[col] = "详情链接"
    elif c in ["作者"]:
        rename_map[col] = "作者"
    elif c in ["出版社"]:
        rename_map[col] = "出版社"
    elif c in ["书名", "标题"]:
        rename_map[col] = "书名"
    elif c in ["折扣率", "折扣"]:
        rename_map[col] = "折扣率"

df = df.rename(columns=rename_map)

required_cols = ["书名", "详情链接", "作者", "出版年份", "出版社", "评论数", "原价", "折后价", "折扣率"]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"缺少字段: {missing_cols}")

df["原始排序"] = range(1, len(df) + 1)
print("✅ 列名标准化完成")

✅ 列名标准化完成


In [73]:
# =========================
# 3. 通用清洗函数
# =========================
def normalize_text(x):
    if pd.isna(x):
        return pd.NA
    x = str(x)
    x = unicodedata.normalize("NFKC", x)
    x = x.strip()
    x = re.sub(r"\s+", " ", x)
    return x if x != "" else pd.NA

marketing_keywords = [
    "正版", "包邮", "满减", "满额减", "电子发票", "放心购买", "新华书店",
    "旗舰店", "官方", "现货", "团购", "赠", "附赠", "视频教学版"
]

def clean_title(title, author=None, publisher=None):
    title = normalize_text(title)
    author = normalize_text(author)
    publisher = normalize_text(publisher)

    if pd.isna(title):
        return pd.NA

    s = str(title)

    # 去掉含营销词的括号内容
    s = re.sub(
        r"[\[\【(（].*?(正版|包邮|满减|满额减|电子发票|放心购买|新华书店|旗舰店|官方|现货|团购|赠|附赠).*?[\]\】)）]",
        "", s, flags=re.IGNORECASE
    )

    # 去ISBN
    s = re.sub(r"ISBN[\s:：-]*\d[\d\-]*", "", s, flags=re.IGNORECASE)

    # 去营销词
    for kw in marketing_keywords:
        s = re.sub(kw, "", s, flags=re.IGNORECASE)

    # 去作者/出版社混入
    if pd.notna(author):
        s = s.replace(str(author), "")
    if pd.notna(publisher):
        s = s.replace(str(publisher), "")

    # 去尾部角色词
    s = re.sub(r"(著|编著|编|主编|译)$", "", s)

    # 去多余符号
    s = re.sub(r"[\【\】\[\]<>《》]", "", s)
    s = re.sub(r"\s+", " ", s).strip(" -_/|·,;；")
    s = re.sub(r"\bpython\b", "Python", s, flags=re.IGNORECASE)

    return s if s else pd.NA

print("✅ 清洗函数定义完成")

✅ 清洗函数定义完成


In [74]:
# =========================
# 4. 文本字段清洗
# =========================
for col in ["书名", "详情链接", "作者", "出版社"]:
    df[col] = df[col].apply(normalize_text)

df["书名_清洗后"] = df.apply(lambda row: clean_title(row["书名"], row["作者"], row["出版社"]), axis=1)
df["作者_标准化"] = df["作者"].apply(normalize_text)
df["出版社_标准化"] = df["出版社"].apply(normalize_text)

print("✅ 文本字段清洗完成")

✅ 文本字段清洗完成


In [75]:
# =========================
# 5. 数值字段标准化
# =========================
for col in ["出版年份", "评论数", "原价", "折后价", "折扣率"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# 年份
df["出版年份_标准化"] = df["出版年份"].where(df["出版年份"].between(1990, 2030), np.nan)
df["出版年份_标准化"] = df["出版年份_标准化"].round().astype("Int64")

# 评论数
df["评论数_标准化"] = df["评论数"].where(df["评论数"] >= 0, np.nan)
df["评论数_标准化"] = df["评论数_标准化"].fillna(0).astype("Int64")

# 价格
df["原价"] = df["原价"].where(df["原价"] > 0, np.nan).round(2)
df["折后价"] = df["折后价"].where(df["折后价"] > 0, np.nan).round(2)

# 折扣率重算
df["折扣率_重算"] = (df["折后价"] / df["原价"]).round(4)
df["折扣率_标准化"] = df["折扣率_重算"].where(df["折扣率_重算"].between(0, 1), np.nan)
df["折扣率百分比"] = (df["折扣率_标准化"] * 100).round(2)

print("✅ 数值字段标准化完成")

✅ 数值字段标准化完成


In [76]:
# =========================
# 6. 缺失值与异常值标记
# =========================
df["作者缺失"] = df["作者_标准化"].isna()
df["年份缺失"] = df["出版年份_标准化"].isna()
df["书名缺失"] = df["书名_清洗后"].isna()
df["评论数_log"] = np.log1p(df["评论数_标准化"].astype(float))

def iqr_flag(series):
    s = pd.to_numeric(series, errors="coerce")
    valid = s.dropna()
    if len(valid) < 4:
        return pd.Series([False] * len(series), index=series.index)

    q1 = valid.quantile(0.25)
    q3 = valid.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return ~((s >= lower) & (s <= upper)) & s.notna()

df["评论数异常_IQR"] = iqr_flag(df["评论数_标准化"].astype(float))
df["原价异常_IQR"] = iqr_flag(df["原价"])
df["折后价异常_IQR"] = iqr_flag(df["折后价"])

# 温莎化
def winsorize_series(series, lower_q=0.01, upper_q=0.99):
    s = pd.to_numeric(series, errors="coerce")
    lower = s.quantile(lower_q)
    upper = s.quantile(upper_q)
    return s.clip(lower=lower, upper=upper)

df["评论数_温莎化"] = winsorize_series(df["评论数_标准化"].astype(float))
df["原价_温莎化"] = winsorize_series(df["原价"])
df["折后价_温莎化"] = winsorize_series(df["折后价"])

print("✅ 缺失值与异常值标记完成")

✅ 缺失值与异常值标记完成


In [77]:
# =========================
# 7. 去重
# =========================
dedup_keys = ["书名_清洗后", "作者_标准化", "出版社_标准化", "出版年份_标准化"]
df["是否重复"] = df.duplicated(subset=dedup_keys, keep="first")

df_clean = (
    df.sort_values("原始排序")
      .drop_duplicates(subset=dedup_keys, keep="first")
      .reset_index(drop=True)
)

df_top50 = df_clean.head(50).copy()

print(f"✅ 去重完成，清洗后记录数: {len(df_clean)}，前50条: {len(df_top50)}")

✅ 去重完成，清洗后记录数: 137，前50条: 50


In [78]:
# =========================
# 8. 清洗日志
# =========================
cleaning_log = pd.DataFrame({
    "指标": [
        "原始记录数", "清洗后记录数", "删除重复数",
        "作者缺失数", "年份缺失数", "书名缺失数",
        "评论数异常(IQR)", "原价异常(IQR)", "折后价异常(IQR)"
    ],
    "数值": [
        len(df), len(df_clean), int(df["是否重复"].sum()),
        int(df_clean["作者缺失"].sum()), int(df_clean["年份缺失"].sum()), int(df_clean["书名缺失"].sum()),
        int(df_clean["评论数异常_IQR"].sum()), int(df_clean["原价异常_IQR"].sum()), int(df_clean["折后价异常_IQR"].sum())
    ]
})

print("✅ 清洗日志生成完成")
cleaning_log

✅ 清洗日志生成完成


,指标,数值
0,原始记录数,154
1,清洗后记录数,137
2,删除重复数,17
3,作者缺失数,15
4,年份缺失数,4
5,书名缺失数,0
6,评论数异常(IQR),19
7,原价异常(IQR),8
8,折后价异常(IQR),35


In [79]:
# =========================
# 9. 生成清洗报告
# =========================
report_text = f"""# 当当网 Python 图书数据清洗报告

## 1. 数据概况
- 原始记录数：{len(df)}
- 清洗后记录数：{len(df_clean)}
- 最终分析样本（前50条）：{len(df_top50)}

## 2. 数据清洗内容
1. 对书名、作者、出版社、链接字段进行了文本标准化处理
2. 对书名字段进行了深度清洗（去营销词、ISBN、作者/出版社混入）
3. 对数值字段进行了格式标准化
4. 对异常值进行了识别（IQR方法）
5. 对重复值进行了处理

## 3. 清洗结果
- 删除重复记录：{int(df["是否重复"].sum())}
- 作者缺失：{int(df_clean["作者缺失"].sum())}
- 出版年份缺失：{int(df_clean["年份缺失"].sum())}
- 书名缺失：{int(df_clean["书名缺失"].sum())}
- 评论数异常值：{int(df_clean["评论数异常_IQR"].sum())}
- 原价异常值：{int(df_clean["原价异常_IQR"].sum())}
- 折后价异常值：{int(df_clean["折后价异常_IQR"].sum())}

## 4. 说明
- 当前数据为按销量排序的搜索结果，不代表真实销量值。
- 评论数极度偏态，建议后续分析中优先使用中位数、分组统计或log变换后的字段。
"""

print("✅ 清洗报告生成完成")
print(report_text)

✅ 清洗报告生成完成
# 当当网 Python 图书数据清洗报告

## 1. 数据概况
- 原始记录数：154
- 清洗后记录数：137
- 最终分析样本（前50条）：50

## 2. 数据清洗内容
1. 对书名、作者、出版社、链接字段进行了文本标准化处理
2. 对书名字段进行了深度清洗（去营销词、ISBN、作者/出版社混入）
3. 对数值字段进行了格式标准化
4. 对异常值进行了识别（IQR方法）
5. 对重复值进行了处理

## 3. 清洗结果
- 删除重复记录：17
- 作者缺失：15
- 出版年份缺失：4
- 书名缺失：0
- 评论数异常值：19
- 原价异常值：8
- 折后价异常值：35

## 4. 说明
- 当前数据为按销量排序的搜索结果，不代表真实销量值。
- 评论数极度偏态，建议后续分析中优先使用中位数、分组统计或log变换后的字段。



In [80]:
# =========================
# 10. 输出清洗后数据
# =========================

# 1. 定义最终需要的字段映射（左边是02清洗后的实际列名，右边是03分析要求的列名）
# 注意：02中已经处理好的列名是 '原价'、'折后价'，而不是 '原价_清洗后'
rename_dict = {
    "书名_清洗后": "书名",
    "作者_标准化": "作者",
    "出版社_标准化": "出版社",
    "出版年份_标准化": "年份",
    "评论数_标准化": "评论数",
    "原价": "原价",
    "折后价": "折后价",
    "折扣率_标准化": "折扣率",
    "详情链接": "链接"
}

# 2. 提取并重命名
df_final = df_clean[list(rename_dict.keys())].rename(columns=rename_dict)

# 3. 补充 03 分析脚本中需要的辅助列
# 增加排名列（基于清洗后的顺序）
df_final.insert(0, '排名', range(1, len(df_final) + 1))

# 补充分类列（如果01爬虫没抓取，这里先给一个默认值，防止03报错）
if '图书分类' not in df_clean.columns:
    df_final['图书分类'] = 'Python图书'
else:
    df_final['图书分类'] = df_clean['图书分类']

# 4. 确保输出目录与 03 脚本的读取路径匹配
# 03 脚本预期读取 'data_clean/dangdang_python_books_clean.csv'
output_dir = output_dir = os.path.join(os.path.dirname(os.path.dirname(file_path)), "data_clean")
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

output_path = os.path.join(output_dir, "python_books_clean.csv")

# 5. 保存文件
df_final.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"✅ 清洗完成！已成功对接分析需求。")
print(f"📂 文件已保存至: {os.path.abspath(output_path)}")
print(f"📊 包含字段: {list(df_final.columns)}")

✅ 清洗完成！已成功对接分析需求。
📂 文件已保存至: /Users/kaka/Documents/中大岭院/课程/数据分析与决策/Proj_homework/data_clean/python_books_clean.csv
📊 包含字段: ['排名', '书名', '作者', '出版社', '年份', '评论数', '原价', '折后价', '折扣率', '链接', '图书分类']
